# Seasonal Agriculture Performance Analysis

**Problem Statement:** Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability and market conditions. As a result, agricultural performance may differ from one season to another. This notebook analyzes a seasonal agriculture dataset to investigate seasonal differences in agricultural performance by identifying meaningful patterns, trends, relationships and variations within the available data.

**Dataset:** 4,000 farm records across 8 states, 8 major crops (Wheat, Rice, Maize, Cotton, Pulses, Groundnut, Chilli, Sugarcane) and three growing seasons - Kharif, Rabi and Zaid.


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", None)
plt.rcParams["figure.facecolor"] = "white"


## 2. Load the Dataset

In [ ]:
df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Check for missing values
df.isnull().sum()


## 3. Data Overview

In [ ]:
print("Seasons:", df['Season'].unique())
print("Crops:", df['Crop'].unique())
print("States:", df['State'].nunique())
print("Irrigation methods:", df['Irrigation_Method'].unique())


## 4. Results

### 4.1 Average Yield & Profit by Season


In [ ]:
season_order = ["Kharif", "Rabi", "Zaid"]

season_summary = df.groupby("Season").agg(
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Rainfall=("Rainfall_mm", "mean"),
    Avg_Temp=("Avg_Temperature_C", "mean"),
    Avg_WaterEff=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_DiseaseRisk=("Disease_Pest_Risk_pct", "mean"),
    Count=("Farm_ID", "count"),
).reindex(season_order).round(2)

season_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#2E83C3", "#42D0A2", "#5FCBEF"]

axes[0].bar(season_order, season_summary["Avg_Yield"], color=colors)
axes[0].set_title("Average Yield by Season", fontweight="bold")
axes[0].set_ylabel("Yield (Tonnes / Hectare)")
for i, v in enumerate(season_summary["Avg_Yield"]):
    axes[0].text(i, v + 0.1, f"{v:.2f}", ha="center", fontweight="bold")

profit_colors = ["#2E946B" if v >= 0 else "#B8544A" for v in season_summary["Avg_Profit"]]
axes[1].bar(season_order, season_summary["Avg_Profit"] / 1000, color=profit_colors)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Average Profit by Season", fontweight="bold")
axes[1].set_ylabel("Avg. Profit (Rs '000 / farm)")
for i, v in enumerate(season_summary["Avg_Profit"] / 1000):
    axes[1].text(i, v + (4 if v >= 0 else -4), f"{v:.0f}K", ha="center",
                 va="bottom" if v >= 0 else "top", fontweight="bold")

plt.tight_layout()
plt.show()


**Insight:** Kharif is the top-performing season (5.64 t/Ha average yield, ~Rs 179K average profit/farm). Zaid season runs at a loss on average (~ -Rs 25K/farm).

### 4.2 Crop-wise Yield Across Seasons

In [ ]:
crop_season = df.groupby(["Crop", "Season"])["Yield_Tonnes_Ha"].mean().unstack()[season_order]
other_crops = [c for c in crop_season.index if c != "Sugarcane"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 2.1]})

axes[0].bar(season_order, crop_season.loc["Sugarcane"], color=colors)
axes[0].set_title("Sugarcane Yield", fontweight="bold")
axes[0].set_ylabel("Tonnes / Hectare")

x = np.arange(len(other_crops))
width = 0.25
for i, (season, c) in enumerate(zip(season_order, colors)):
    axes[1].bar(x + (i - 1) * width, crop_season.loc[other_crops, season], width=width, label=season, color=c)
axes[1].set_xticks(x)
axes[1].set_xticklabels(other_crops)
axes[1].set_title("Other Crops - Yield by Season", fontweight="bold")
axes[1].set_ylabel("Tonnes / Hectare")
axes[1].legend()

plt.tight_layout()
plt.show()


**Insight:** Sugarcane yields (38-53 t/Ha) dwarf every other crop, and every crop's yield declines from Kharif to Rabi to Zaid.

### 4.3 Average Profit by Crop

In [ ]:
profit_crop = df.groupby("Crop")["Profit_INR"].mean().sort_values() / 1000

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ["#2E946B" if v >= 0 else "#B8544A" for v in profit_crop]
bars = ax.barh(profit_crop.index, profit_crop.values, color=bar_colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Average Profit per Farm by Crop", fontweight="bold")
ax.set_xlabel("Average Profit (Rs '000 per farm)")

for b, v in zip(bars, profit_crop.values):
    ax.annotate(f"{v:,.0f}K", xy=(v, b.get_y() + b.get_height() / 2),
                xytext=(8 if v >= 0 else -8, 0), textcoords="offset points",
                va="center", ha="left" if v >= 0 else "right", fontweight="bold")

plt.tight_layout()
plt.show()


**Insight:** Sugarcane, Chilli and Cotton are the most profitable crops on average; Wheat, Rice and Maize run at a loss.

### 4.4 Water-Use Efficiency vs. Yield

In [ ]:
corr = df["Water_Efficiency_t_per_1000m3"].corr(df["Yield_Tonnes_Ha"])
print(f"Correlation (r) = {corr:.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df["Water_Efficiency_t_per_1000m3"], df["Yield_Tonnes_Ha"], s=18, alpha=0.4, color="#2E83C3")

z = np.polyfit(df["Water_Efficiency_t_per_1000m3"], df["Yield_Tonnes_Ha"], 1)
xs = np.linspace(df["Water_Efficiency_t_per_1000m3"].min(), df["Water_Efficiency_t_per_1000m3"].max(), 100)
ax.plot(xs, np.polyval(z, xs), color="#B8544A", linewidth=2, label="Trend line")

ax.set_title("Water-Use Efficiency vs. Crop Yield", fontweight="bold")
ax.set_xlabel("Water Efficiency (Tonnes produced / 1000 m3 water used)")
ax.set_ylabel("Yield (Tonnes / Hectare)")
ax.legend()
ax.text(0.03, 0.93, f"r = {corr:.2f}", transform=ax.transAxes, fontweight="bold",
        bbox=dict(boxstyle="round", fc="white", ec="gray"))

plt.tight_layout()
plt.show()


**Insight:** Water-use efficiency is the strongest driver of yield in this dataset (r = 0.92) - stronger than rainfall or temperature alone.

### 4.5 Irrigation Method Usage & Disease/Pest Risk by Season

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

irr_counts = df["Irrigation_Method"].value_counts()
axes[0].pie(irr_counts.values, labels=irr_counts.index, autopct="%1.0f%%",
            colors=["#2E83C3", "#42D0A2", "#5FCBEF", "#2E946B"], startangle=90,
            wedgeprops=dict(width=0.42, edgecolor="white"))
axes[0].set_title("Irrigation Method Usage", fontweight="bold")

risk = df.groupby("Season")["Disease_Pest_Risk_pct"].mean().reindex(season_order)
axes[1].bar(season_order, risk.values, color=colors)
axes[1].set_title("Avg. Disease/Pest Risk by Season", fontweight="bold")
axes[1].set_ylabel("Risk (%)")
for i, v in enumerate(risk.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()


**Insight:** Flood irrigation is the most common method (33%) despite generally lower water efficiency than Drip or Sprinkler. Disease/pest risk peaks in the Kharif season (54.5%), likely linked to monsoon humidity.

## 5. Conclusions

- **Kharif** is the strongest season overall - highest yield (5.64 t/Ha) and highest average profit (~Rs 179K/farm).
- **Zaid** season is loss-making on average (~ -Rs 25K/farm) and has the lowest yield of the three seasons.
- **Sugarcane, Chilli, and Cotton** are the most profitable crops; **Wheat, Rice, and Maize** run at a loss on average.
- **Water-use efficiency** is the single strongest driver of yield (r = 0.92) - more influential than rainfall or temperature.
- **Flood irrigation** remains the dominant method (33% of farms) despite being less water-efficient than Drip or Sprinkler systems.
- **Disease/pest risk** is highest in Kharif (54.5%), consistent with monsoon-season humidity favoring pest and disease pressure.

## 6. Future Scope

- Build ML models to predict yield and profit before the season starts.
- Add real-time weather and market-price feeds for live recommendations.
- Extend the dataset to more states, crops, and years for stronger trend detection.
- Develop a farmer-facing dashboard/app for season-wise crop advisory.
- Investigate the Zaid-season losses in depth and pilot low-cost drip irrigation.
- Integrate satellite/remote-sensing data for soil and crop-health monitoring.
